Eliminar potential cuando exista un customer con el mismo Global Parent + Ubicación

In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Ivan.DeLaFuente\OneDrive - Quaker Houghton\Documents\QH WorkDay\Junio 2026\T&P Market\T&P_Market.csv')

In [2]:
# Create unique location
df['Ship To Country'] = df['Ship To Country'].fillna('')
df['Ship To State']   = df['Ship To State'].fillna('')
df['Ship To City']    = df['Ship To City'].fillna('')

df['Location'] = (
    df['Ship To Country'] + '|' +
    df['Ship To State'] + '|' +
    df['Ship To City']
)

df["Location"] = df["Location"].str.upper()

In [3]:
# Step 1: Create customer set
customer_pairs = df[df['Status'] == 'Customer'][['Global Parent', 'Location']]
customer_set = set(zip(customer_pairs['Global Parent'], customer_pairs['Location']))

# Step 2: Create Action column
def classify(row):
    
    key = (row['Global Parent'], row['Location'])
    
    # Case 1: Customer → always keep
    if row['Status'] == 'Customer':
        return "Keep - Customer"
    
    # Case 2: Potential + customer exists → remove
    if key in customer_set:
        return "Remove - Duplicate of Customer"
    
    # Case 3: Potential (no customer) → keep for now
    return "Keep - Potential (no customer)"

df['Action'] = df.apply(classify, axis=1)

In [4]:
# Only potentials that survived
potential_mask = df['Status'] == "Potential"

# Mark duplicates
df.loc[potential_mask, 'Duplicate Flag'] = (
    df[potential_mask]
    .duplicated(subset=['Global Parent', 'Location'], keep='first')
)

# Update Action for duplicates
df.loc[
    (df['Duplicate Flag'] == True),
    'Action'
] = "Remove - Duplicate Potential"

In [ ]:
# Final df
final_df = df[df['Action'].str.startswith("Keep")]

final_df = final_df.drop(columns=['Location', 'Action', 'Duplicate Flag'])


